# Model 2 — Legal NLI (XLM-RoBERTa-large) — All-in-One

This notebook fine-tunes a **multilingual Natural Language Inference** model on legal text.
End-to-end pipeline:

| # | Section | What happens |
|---|---------|-------------|
| 1 | **Setup** | Install deps, mount Drive, set paths |
| 2 | **Load raw data** | Read `nli_triples.parquet` from Drive |
| 3 | **Cleaning** | Drop nulls, strip whitespace, normalise label strings |
| 4 | **Deduplication** | Remove duplicate (premise, hypothesis) pairs |
| 5 | **EDA & report figures** | 14 plots saved for the PFE report |
| 6 | **Train / val split** | Stratified 90 / 10 by label |
| 7 | **Tokenisation** | XLM-RoBERTa-large tokenizer, max_len=256 |
| 8 | **Fine-tuning** | 3-class head, AdamW, linear warmup, early stopping |
| 9 | **Evaluation** | Accuracy, macro-F1, per-class F1, confusion matrix |
|10 | **Save artefacts** | Model, tokenizer, metrics JSON, all plots |

> **GPU required.** Use *Runtime → Change runtime type → T4 GPU* before running.


---
## Section 1 — Setup


In [ ]:
# Install / upgrade required packages (run once per Colab session)
# scikit-learn, tqdm, fsspec are intentionally unpinned — Colab pre-installs
# hdbscan, umap-learn, gcsfs, dataproc which require newer versions of these.
!pip install -q --upgrade \
    transformers==4.41.2 \
    datasets==2.20.0 \
    accelerate==0.31.0 \
    scikit-learn \
    pandas==2.2.2 \
    pyarrow==16.1.0 \
    matplotlib==3.9.0 \
    seaborn==0.13.2 \
    tqdm
print('All packages ready.')

In [ ]:
from __future__ import annotations

import json
import os
import random
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
)
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── EDIT THIS if your file is in a subfolder ──────────────────────────────
PARQUET_PATH = Path('/content/drive/MyDrive/nli_triples.parquet')
# ───────────────────────────────────────────────────────────────────────────

OUTPUT_DIR  = Path('/content/drive/MyDrive/nli_model_output')
MODEL_DIR   = OUTPUT_DIR / 'xlmr_legal_nli'
FIGURES_DIR = OUTPUT_DIR / 'figures'

for d in [OUTPUT_DIR, MODEL_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Parquet :', PARQUET_PATH)
print('Output  :', OUTPUT_DIR)

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f'File not found: {PARQUET_PATH}\n'
        'Upload nli_triples.parquet to your Google Drive root and re-run.'
    )


---
## Section 2 — Load Raw Data


In [ ]:
df_raw = pd.read_parquet(PARQUET_PATH)

print(f'Raw shape  : {df_raw.shape}')
print(f'Columns    : {list(df_raw.columns)}')
print('\nLabel distribution:')
print(df_raw['label'].value_counts().to_string())
print('\nPremise language distribution:')
print(df_raw['premise_lang'].value_counts().to_string())
print('\nHypothesis language distribution:')
print(df_raw['hypothesis_lang'].value_counts().to_string())
print(f"\nCross-lingual rows : {int(df_raw['is_cross_lingual'].sum())} "
      f"({100*df_raw['is_cross_lingual'].mean():.1f}%)")
df_raw.head(3)


---
## Section 3 — Cleaning


In [ ]:
df = df_raw.copy()
rows_before_clean = len(df)

# 3a. Report nulls/empty BEFORE
print('=== NULL / EMPTY CHECK (before) ===')
for col in ['premise', 'hypothesis', 'label', 'premise_lang', 'hypothesis_lang']:
    null_n  = df[col].isnull().sum()
    empty_n = (df[col].astype(str).str.strip() == '').sum()
    print(f'  {col:22s}  nulls={null_n}  empty={empty_n}')

# 3b. Strip whitespace, drop empty premise / hypothesis
df['premise']    = df['premise'].astype(str).str.strip()
df['hypothesis'] = df['hypothesis'].astype(str).str.strip()
df = df[
    (df['premise'] != '') & (df['hypothesis'] != '') &
    (df['premise'] != 'nan') & (df['hypothesis'] != 'nan')
].copy()

# 3c. Normalise label to lowercase
df['label'] = df['label'].astype(str).str.strip().str.lower()
valid_labels = {'entailed', 'neutral', 'contradicted'}
bad_label_mask = ~df['label'].isin(valid_labels)
if bad_label_mask.sum():
    print(f'\nDropping {bad_label_mask.sum()} rows with unrecognised labels:')
    print(df[bad_label_mask]['label'].value_counts().to_string())
df = df[df['label'].isin(valid_labels)].copy()

# 3d. Map labels to integer ids  (contradicted=0, neutral=1, entailed=2)
LABEL2ID = {'contradicted': 0, 'neutral': 1, 'entailed': 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
df['label_id'] = df['label'].map(LABEL2ID)

rows_after_clean = len(df)
print(f'\nRows before clean : {rows_before_clean:,}')
print(f'Rows after  clean : {rows_after_clean:,}  '
      f'(dropped {rows_before_clean - rows_after_clean})')

print('\n=== NULL / EMPTY CHECK (after) ===')
for col in ['premise', 'hypothesis', 'label']:
    null_n  = df[col].isnull().sum()
    empty_n = (df[col].astype(str).str.strip() == '').sum()
    print(f'  {col:22s}  nulls={null_n}  empty={empty_n}')


---
## Section 4 — Deduplication


In [ ]:
rows_before_dedup = len(df)

# Exact (premise, hypothesis) duplicates
exact_dups = df.duplicated(subset=['premise', 'hypothesis']).sum()
print(f'Exact duplicate (premise, hypothesis) pairs : {exact_dups}')
df = df.drop_duplicates(subset=['premise', 'hypothesis']).copy()

# Same hypothesis reused across different premises
dup_hyp = df['hypothesis'].duplicated().sum()
print(f'Duplicate hypothesis strings               : {dup_hyp}')
df = df.drop_duplicates(subset=['hypothesis']).copy()

rows_after_dedup = len(df)
print(f'\nRows before dedup : {rows_before_dedup:,}')
print(f'Rows after  dedup : {rows_after_dedup:,}  '
      f'(dropped {rows_before_dedup - rows_after_dedup})')

print('\n=== Label balance after dedup ===')
for lbl, cnt in df['label'].value_counts().items():
    print(f'  {lbl:15s} : {cnt:,}  ({100*cnt/len(df):.1f}%)')

print('\n=== Premise language ===')
print(df['premise_lang'].value_counts().to_string())

xling = int(df['is_cross_lingual'].sum())
print(f'\nCross-lingual : {xling} / {len(df)}  ({100*xling/len(df):.1f}%)')


---
## Section 5 — EDA & Report Figures

All figures are saved to `OUTPUT_DIR/figures/` — include them in your PFE report.


In [ ]:
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

PALETTE = {'contradicted': '#e53e3e', 'neutral': '#d69e2e', 'entailed': '#38a169'}
LANG_COLORS = {'en': '#2b6cb0', 'ar': '#d69e2e', 'fr': '#805ad5', 'de': '#2c7a7b'}
figure_paths: list[str] = []

def save_fig(name: str) -> str:
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    figure_paths.append(str(path))
    print(f'  saved -> {path.name}')
    return str(path)

print('EDA helpers ready.')


In [ ]:
# Figure 1: Label distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['label'].value_counts().reindex(['entailed', 'neutral', 'contradicted'])
bars = ax.bar(counts.index, counts.values,
              color=[PALETTE[l] for l in counts.index], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Figure 1 — NLI Label Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Label'); ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.15)
save_fig('nli_01_label_distribution.png'); plt.show()


In [ ]:
# Figure 2: Premise language distribution
fig, ax = plt.subplots(figsize=(7, 4))
plang = df['premise_lang'].value_counts()
bars = ax.bar(plang.index, plang.values,
              color=[LANG_COLORS.get(l, '#718096') for l in plang.index], edgecolor='white')
for bar, val in zip(bars, plang.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(val), ha='center', va='bottom', fontsize=10)
ax.set_title('Figure 2 — Premise Language Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Language'); ax.set_ylabel('Count')
save_fig('nli_02_premise_lang.png'); plt.show()


In [ ]:
# Figure 3: Hypothesis language distribution
fig, ax = plt.subplots(figsize=(7, 4))
hlang = df['hypothesis_lang'].value_counts()
bars = ax.bar(hlang.index, hlang.values,
              color=[LANG_COLORS.get(l, '#718096') for l in hlang.index], edgecolor='white')
for bar, val in zip(bars, hlang.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(val), ha='center', va='bottom', fontsize=10)
ax.set_title('Figure 3 — Hypothesis Language Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Language'); ax.set_ylabel('Count')
save_fig('nli_03_hypothesis_lang.png'); plt.show()


In [ ]:
# Figure 4: Cross-lingual vs monolingual breakdown
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

xling_counts = df['is_cross_lingual'].value_counts().reindex([True, False], fill_value=0)
axes[0].pie(xling_counts.values, labels=['Cross-lingual', 'Monolingual'],
            autopct='%1.1f%%', colors=['#2b6cb0', '#cbd5e0'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Cross-lingual Split', fontweight='bold')

cbl = df.groupby(['premise_lang', 'is_cross_lingual']).size().unstack(fill_value=0)
cbl = cbl.rename(columns={True: 'Cross-lingual', False: 'Monolingual'})
for c in ['Monolingual', 'Cross-lingual']:
    if c not in cbl.columns:
        cbl[c] = 0
cbl[['Monolingual', 'Cross-lingual']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#cbd5e0', '#2b6cb0'], edgecolor='white')
axes[1].set_title('Cross-lingual by Premise Language', fontweight='bold')
axes[1].set_xlabel('Premise Language'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

fig.suptitle('Figure 4 — Cross-lingual Pair Analysis', fontsize=13, fontweight='bold')
save_fig('nli_04_cross_lingual.png'); plt.show()


In [ ]:
# Figure 5: Text length distributions
df['premise_len']    = df['premise'].str.len()
df['hypothesis_len'] = df['hypothesis'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cap_p = int(df['premise_len'].quantile(0.99))
axes[0].hist(df['premise_len'].clip(upper=cap_p), bins=40, color='#2b6cb0', edgecolor='white')
axes[0].set_title('Premise Length (chars, 99th pct cap)', fontweight='bold')
axes[0].set_xlabel('Characters'); axes[0].set_ylabel('Count')
axes[0].axvline(df['premise_len'].median(), color='#e53e3e', linestyle='--',
                label=f"median={int(df['premise_len'].median())}")
axes[0].legend()

cap_h = int(df['hypothesis_len'].quantile(0.99))
axes[1].hist(df['hypothesis_len'].clip(upper=cap_h), bins=40, color='#38a169', edgecolor='white')
axes[1].set_title('Hypothesis Length (chars, 99th pct cap)', fontweight='bold')
axes[1].set_xlabel('Characters'); axes[1].set_ylabel('Count')
axes[1].axvline(df['hypothesis_len'].median(), color='#e53e3e', linestyle='--',
                label=f"median={int(df['hypothesis_len'].median())}")
axes[1].legend()

fig.suptitle('Figure 5 — Text Length Distributions', fontsize=13, fontweight='bold')
save_fig('nli_05_text_lengths.png'); plt.show()


In [ ]:
# Figure 6: Label x premise language heatmap
ct = pd.crosstab(df['premise_lang'], df['label'])[['entailed', 'neutral', 'contradicted']]
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8})
ax.set_title('Figure 6 — Label x Premise Language', fontsize=13, fontweight='bold')
ax.set_xlabel('Label'); ax.set_ylabel('Premise Language')
save_fig('nli_06_label_x_lang_heatmap.png'); plt.show()


In [ ]:
# Figure 7: Language-pair matrix (premise_lang -> hypothesis_lang)
lp = pd.crosstab(df['premise_lang'], df['hypothesis_lang'])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(lp, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Figure 7 — Language-Pair Matrix (Premise -> Hypothesis)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Hypothesis Language'); ax.set_ylabel('Premise Language')
save_fig('nli_07_lang_pair_matrix.png'); plt.show()


In [ ]:
# Figure 8: Country distribution
fig, ax = plt.subplots(figsize=(7, 4))
cntry = df['premise_country'].value_counts()
colors = ['#2b6cb0', '#d69e2e', '#38a169', '#805ad5']
bars = ax.bar(cntry.index, cntry.values,
              color=colors[:len(cntry)], edgecolor='white')
for bar, val in zip(bars, cntry.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=11)
ax.set_title('Figure 8 — Premise Country Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Country'); ax.set_ylabel('Count')
save_fig('nli_08_country_distribution.png'); plt.show()


In [ ]:
# Figure 9: Code family distribution (top 8)
fig, ax = plt.subplots(figsize=(9, 4))
cf = (df['premise_code_family'].replace('', 'unknown')
        .fillna('unknown').value_counts().head(8))
cf.sort_values().plot(kind='barh', ax=ax, color='#2c7a7b', edgecolor='white')
ax.set_title('Figure 9 — Top Code Families (Premises)', fontsize=13, fontweight='bold')
ax.set_xlabel('Count'); ax.set_ylabel('Code Family')
for patch in ax.patches:
    ax.text(patch.get_width() + 2, patch.get_y() + patch.get_height()/2,
            str(int(patch.get_width())), va='center', fontsize=9)
save_fig('nli_09_code_family.png'); plt.show()


In [ ]:
# Figure 10: Dataset size at each preparation stage
stages     = ['Raw', 'After\nCleaning', 'After\nDedup']
stage_rows = [rows_before_clean, rows_after_clean, rows_after_dedup]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(stages, stage_rows,
              color=['#718096', '#3182ce', '#38a169'],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, stage_rows):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Figure 10 — Dataset Size at Each Preparation Stage',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Row Count')
ax.set_ylim(0, max(stage_rows) * 1.15)
save_fig('nli_10_cleaning_stages.png'); plt.show()
print(f'\n{len(figure_paths)} figures saved so far.')


---
## Section 6 — Train / Validation Split


In [ ]:
df_clean = df.reset_index(drop=True)

train_df, val_df = train_test_split(
    df_clean,
    test_size=0.10,
    random_state=SEED,
    stratify=df_clean['label_id'],
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train rows : {len(train_df):,}')
print(f'Val   rows : {len(val_df):,}')
print('\nTrain label dist:')
print(train_df['label'].value_counts().to_string())
print('\nVal label dist:')
print(val_df['label'].value_counts().to_string())


---
## Section 7 — Tokenisation


In [ ]:
MODEL_NAME = 'xlm-roberta-large'
MAX_LEN    = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded : {MODEL_NAME}')
print(f'Vocab size       : {tokenizer.vocab_size:,}')

# Token length sanity check on a 500-row sample
sample_enc = tokenizer(
    df_clean['premise'].tolist()[:500],
    df_clean['hypothesis'].tolist()[:500],
    truncation=True, max_length=MAX_LEN, padding=False,
    return_attention_mask=False, return_token_type_ids=False,
)
token_lens = [len(ids) for ids in sample_enc['input_ids']]
pct_trunc  = sum(l == MAX_LEN for l in token_lens) / len(token_lens) * 100
print(f'\nToken lengths (n=500): min={min(token_lens)}  '
      f'median={int(np.median(token_lens))}  max={max(token_lens)}')
print(f'Rows hitting MAX_LEN ({MAX_LEN}): {pct_trunc:.1f}%')


In [ ]:
class NLIDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_len: int) -> None:
        self.premises   = dataframe['premise'].tolist()
        self.hypotheses = dataframe['hypothesis'].tolist()
        self.labels     = dataframe['label_id'].tolist()
        self.tokenizer  = tokenizer
        self.max_len    = max_len

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tokenizer(
            self.premises[idx], self.hypotheses[idx],
            max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long),
        }


BATCH_SIZE = 8  # safe for T4 VRAM with xlm-roberta-large + MAX_LEN=256

train_dataset = NLIDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = NLIDataset(val_df,   tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')


---
## Section 8 — Fine-tuning


In [ ]:
NUM_LABELS   = 3
NUM_EPOCHS   = 4
LR           = 1e-5
WARMUP_RATIO = 0.06

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(DEVICE)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Model         : {MODEL_NAME}')
print(f'Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print(f'Epochs        : {NUM_EPOCHS}')
print(f'Total steps   : {total_steps:,}')
print(f'Warmup steps  : {warmup_steps}')
print(f'LR            : {LR}')
print(f'Batch size    : {BATCH_SIZE}')


In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, macro_f1, all_labels, all_preds


history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = -1.0
best_epoch  = -1
patience    = 2
no_improve  = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [train]', leave=False)
    for step, batch in enumerate(pbar, 1):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += outputs.loss.item()
        pbar.set_postfix({'loss': f'{running_loss/step:.4f}'})

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc, val_f1, _, _ = evaluate_model(model, val_loader, DEVICE)

    history['epoch'].append(epoch)
    history['train_loss'].append(round(train_loss, 4))
    history['val_loss'].append(round(val_loss, 4))
    history['val_acc'].append(round(val_acc, 4))
    history['val_f1'].append(round(val_f1, 4))

    print(f'Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | '
          f'val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_macro_f1={val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        no_improve  = 0
        model.save_pretrained(str(MODEL_DIR))
        tokenizer.save_pretrained(str(MODEL_DIR))
        print(f'  >> Best model saved (val_macro_f1={best_val_f1:.4f})')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{patience})')
        if no_improve >= patience:
            print('  Early stopping triggered.')
            break

print(f'\nDone. Best epoch={best_epoch}, best val_macro_f1={best_val_f1:.4f}')


---
## Section 9 — Evaluation


In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(DEVICE)
print(f'Best model loaded from {MODEL_DIR}')


In [ ]:
_, final_acc, final_f1, y_true, y_pred = evaluate_model(best_model, val_loader, DEVICE)
label_names = [ID2LABEL[i] for i in sorted(ID2LABEL)]

print(f'=== Final Validation Results (best epoch={best_epoch}) ===')
print(f'Accuracy  : {final_acc:.4f}')
print(f'Macro F1  : {final_f1:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))


In [ ]:
# Figure 11: Training curves
hist_df = pd.DataFrame(history)
epochs  = hist_df['epoch'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, hist_df['train_loss'], 'o-', color='#2b6cb0', label='Train')
axes[0].plot(epochs, hist_df['val_loss'],   's--', color='#e53e3e', label='Val')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy')
axes[0].legend()
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[1].plot(epochs, hist_df['val_acc'], 'o-', color='#38a169')
axes[1].set_title('Val Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[2].plot(epochs, hist_df['val_f1'], 'o-', color='#805ad5')
axes[2].set_title('Val Macro F1', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Macro F1')
axes[2].set_ylim(0, 1)
axes[2].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle('Figure 11 — Training Curves', fontsize=13, fontweight='bold')
save_fig('nli_11_training_curves.png'); plt.show()


In [ ]:
# Figure 12: Confusion matrix
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

annot = np.array([[f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)'
                    for j in range(cm.shape[1])]
                   for i in range(cm.shape[0])])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
            xticklabels=label_names, yticklabels=label_names,
            linewidths=0.5, linecolor='white')
ax.set_title('Figure 12 — Confusion Matrix (Validation Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
save_fig('nli_12_confusion_matrix.png'); plt.show()


In [ ]:
# Figure 13: Per-class F1
per_class_f1 = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(label_names, per_class_f1,
              color=[PALETTE[l] for l in label_names], edgecolor='white')
for bar, val in zip(bars, per_class_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.axhline(final_f1, color='#718096', linestyle='--', linewidth=1.2,
           label=f'Macro F1 = {final_f1:.3f}')
ax.set_ylim(0, 1.1)
ax.set_title('Figure 13 — Per-Class F1 Score (Validation Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Label'); ax.set_ylabel('F1 Score')
ax.legend()
save_fig('nli_13_per_class_f1.png'); plt.show()


In [ ]:
# Figure 14: Per-language macro F1
val_df_eval = val_df.reset_index(drop=True).copy()
val_df_eval['pred'] = y_pred

lang_f1_rows = []
for lang in sorted(val_df_eval['premise_lang'].unique()):
    sub = val_df_eval[val_df_eval['premise_lang'] == lang]
    if len(sub) < 3:
        continue
    lf1 = f1_score(sub['label_id'], sub['pred'], average='macro', zero_division=0)
    lang_f1_rows.append({'lang': lang, 'macro_f1': round(lf1, 4), 'n_rows': len(sub)})

lang_f1_df = pd.DataFrame(lang_f1_rows)
print('Per-language validation Macro F1:')
print(lang_f1_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(lang_f1_df['lang'], lang_f1_df['macro_f1'],
              color=[LANG_COLORS.get(l, '#718096') for l in lang_f1_df['lang']],
              edgecolor='white')
for bar, row in zip(bars, lang_f1_df.itertuples(index=False)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{row.macro_f1:.3f}\n(n={row.n_rows})',
            ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_title('Figure 14 — Per-Language Macro F1 (Validation Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Premise Language'); ax.set_ylabel('Macro F1')
save_fig('nli_14_per_lang_f1.png'); plt.show()


---
## Section 10 — Save Artefacts


In [ ]:
cr_dict = classification_report(
    y_true, y_pred, target_names=label_names, digits=4, output_dict=True
)

metrics_payload = {
    'generated_at_utc' : datetime.now(timezone.utc).isoformat(),
    'model'            : MODEL_NAME,
    'best_epoch'       : best_epoch,
    'num_epochs_run'   : len(history['epoch']),
    'max_len'          : MAX_LEN,
    'batch_size'       : BATCH_SIZE,
    'learning_rate'    : LR,
    'data': {
        'raw_rows'         : rows_before_clean,
        'after_clean_rows' : rows_after_clean,
        'after_dedup_rows' : rows_after_dedup,
        'train_rows'       : len(train_df),
        'val_rows'         : len(val_df),
        'label2id'         : LABEL2ID,
    },
    'training_history'     : history,
    'val_accuracy'         : round(final_acc, 4),
    'val_macro_f1'         : round(final_f1, 4),
    'per_class_f1'         : {label_names[i]: round(float(per_class_f1[i]), 4)
                               for i in range(len(label_names))},
    'classification_report': cr_dict,
    'per_language_f1'      : lang_f1_df.to_dict(orient='records'),
    'confusion_matrix'     : cm.tolist(),
    'figures'              : figure_paths,
}

metrics_path = OUTPUT_DIR / 'nli_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

clean_parquet_path = OUTPUT_DIR / 'nli_triples_clean.parquet'
df_clean.to_parquet(clean_parquet_path, index=False)

print('=== Saved artefacts ===')
print(f'  metrics    : {metrics_path}')
print(f'  clean data : {clean_parquet_path}')
print(f'  model dir  : {MODEL_DIR}')
for fp in figure_paths:
    print(f'  figure     : {Path(fp).name}')
print(f'\nTotal figures: {len(figure_paths)}')


In [ ]:
# Final summary table
summary = pd.DataFrame([
    {'metric': 'Val Accuracy',      'value': f'{final_acc:.4f}'},
    {'metric': 'Val Macro F1',       'value': f'{final_f1:.4f}'},
    {'metric': 'F1 — contradicted',  'value': f'{per_class_f1[0]:.4f}'},
    {'metric': 'F1 — neutral',       'value': f'{per_class_f1[1]:.4f}'},
    {'metric': 'F1 — entailed',      'value': f'{per_class_f1[2]:.4f}'},
    {'metric': 'Best epoch',          'value': str(best_epoch)},
    {'metric': 'Train rows',          'value': str(len(train_df))},
    {'metric': 'Val rows',            'value': str(len(val_df))},
    {'metric': 'Model',               'value': MODEL_NAME},
])
print(summary.to_string(index=False))
